# Notebook 03: Preprocessing Pipeline
## TSFM Industrial PdM Benchmark - ICATH Conference

**Author:** Yassire Ammouri  
**Purpose:** Run SCADA-optimized preprocessing on all datasets  
**Key Features:**
- Chronological splits (no data leakage)
- Per-sensor-family normalization
- Kalman-based imputation
- Health indicator extraction for RUL

### Step 1: Setup and Imports

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# Add project source to path
PROJECT_DIR = Path("/content/tsfm-pdm-benchmark")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from data.preprocessing import SCADAPreprocessor

# Paths
RAW_DIR = PROJECT_DIR / "data/raw"
PROCESSED_DIR = PROJECT_DIR / "data/processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data: {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")

### Step 2: Load Configuration

In [ ]:
# Load config
with open(PROJECT_DIR / "config/config.yaml", 'r') as f:
    config = yaml.safe_load(f)

preprocessing_config = config['preprocessing']
print("Preprocessing Configuration:")
print(f"  Lookback window: {preprocessing_config['lookback_window']}")
print(f"  Forecast horizon: {preprocessing_config['forecast_horizon']}")
print(f"  Train ratio: {preprocessing_config['train_ratio']}")
print(f"  Val ratio: {preprocessing_config['val_ratio']}")
print(f"  Test ratio: {1.0 - preprocessing_config['train_ratio'] - preprocessing_config['val_ratio']:.2f}")
print(f"  Normalization: {preprocessing_config['normalization']}")
print(f"  Imputation: {preprocessing_config['imputation']}")

### Step 3: Initialize Preprocessor

In [ ]:
preprocessor = SCADAPreprocessor(
    lookback=preprocessing_config['lookback_window'],
    horizon=preprocessing_config['forecast_horizon'],
    train_ratio=preprocessing_config['train_ratio'],
    val_ratio=preprocessing_config['val_ratio'],
    normalization=preprocessing_config['normalization']
)

print("SCADA Preprocessor initialized!")

### Step 4: Process C-MAPSS Dataset
**Priority: HIGH** - Process all 4 subsets (FD001-FD004)

In [ ]:
cmapss_raw = RAW_DIR / "cmapss"

if cmapss_raw.exists() and (cmapss_raw / "train_FD001.txt").exists():
    print("Processing C-MAPSS Dataset...")
    print("=" * 60)
    
    cmapss_results = {}
    
    for subset in ["FD001", "FD002", "FD003", "FD004"]:
        try:
            print(f"\nProcessing {subset}...")
            data = preprocessor.process_cmapss(cmapss_raw, subset)
            
            # Save processed data
            output_dir = PROCESSED_DIR / "cmapss" / subset
            output_dir.mkdir(parents=True, exist_ok=True)
            preprocessor.save_processed(data, output_dir)
            
            cmapss_results[subset] = {
                "train_shape": list(data['train_X'].shape),
                "val_shape": list(data['val_X'].shape),
                "test_shape": list(data['test_X'].shape),
                "num_channels": data['num_channels'],
                "task": data['task']
            }
            
        except Exception as e:
            print(f"Error processing {subset}: {e}")
    
    print("\n" + "=" * 60)
    print("C-MAPSS Processing Summary:")
    for subset, info in cmapss_results.items():
        print(f"  {subset}: Train={info['train_shape']}, Test={info['test_shape']}")
        
else:
    print("C-MAPSS not found! Please run Notebook 02 first.")

### Step 5: Process Generic CSV Datasets
For datasets that don't have a specific processor, use the generic CSV processor

In [ ]:
def process_generic_dataset(name, raw_path, task="forecasting", timestamp_col=None, target_col=None):
    """Process a generic CSV dataset"""
    print(f"\nProcessing {name}...")
    print("=" * 60)
    
    if not raw_path.exists():
        print(f"  {name} not found! Skipping.")
        return None
    
    # Find CSV files
    csv_files = list(raw_path.glob("*.csv"))
    if not csv_files:
        csv_files = list(raw_path.glob("*.txt"))
    
    if not csv_files:
        print(f"  No CSV/TXT files found in {raw_path}")
        return None
    
    print(f"  Found {len(csv_files)} file(s)")
    
    # Process first file as example
    try:
        data = preprocessor.process_generic_csv(
            csv_files[0],
            timestamp_col=timestamp_col,
            target_col=target_col,
            task=task
        )
        
        # Save
        output_dir = PROCESSED_DIR / name
        output_dir.mkdir(parents=True, exist_ok=True)
        preprocessor.save_processed(data, output_dir)
        
        print(f"  Train: {data['train_X'].shape}")
        print(f"  Val: {data['val_X'].shape}")
        print(f"  Test: {data['test_X'].shape}")
        
        return data
        
    except Exception as e:
        print(f"  Error: {e}")
        return None

In [ ]:
# Process Wind SCADA (if available)
process_generic_dataset(
    "wind_scada",
    RAW_DIR / "wind_scada",
    task="forecasting",
    timestamp_col=None
)

In [ ]:
# Process PHM Milling (if available)
process_generic_dataset(
    "phm_milling",
    RAW_DIR / "phm_milling",
    task="anomaly_detection"
)

In [ ]:
# Process MIMII (if available)
process_generic_dataset(
    "mimii",
    RAW_DIR / "mimii",
    task="anomaly_detection"
)

### Step 6: Verify Processed Data

In [ ]:
print("=" * 60)
print("PROCESSED DATA SUMMARY")
print("=" * 60)

processed_datasets = list(PROCESSED_DIR.glob("*/processed_data.pt"))
processed_datasets += list(PROCESSED_DIR.glob("/*/processed_data.pt"))

if not processed_datasets:
    print("No processed datasets found!")
else:
    for p in sorted(set(processed_datasets)):
        try:
            data = torch.load(p)
            print(f"\n{p.relative_to(PROCESSED_DIR)}:")
            print(f"  Task: {data.get('task', 'unknown')}")
            print(f"  Channels: {data.get('num_channels', 'unknown')}")
            print(f"  Train: {data['train_X'].shape}")
            print(f"  Val: {data['val_X'].shape}")
            print(f"  Test: {data['test_X'].shape}")
            print(f"  Lookback: {data.get('lookback', 'unknown')}")
            print(f"  Horizon: {data.get('horizon', 'unknown')}")
        except Exception as e:
            print(f"Error loading {p}: {e}")

print("\n" + "=" * 60)
print("Preprocessing complete! Ready for experiments.")
print("Next: Run Notebook 04 (Zero-Shot Experiments)")

### Step 7: Visualize Sample Data (Optional)

In [ ]:
# Visualize a sample from C-MAPSS FD001
cmapss_processed = PROCESSED_DIR / "cmapss" / "FD001" / "processed_data.pt"

if cmapss_processed.exists():
    data = torch.load(cmapss_processed)
    
    # Plot first training sample
    sample = data['train_X'][0].numpy()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.flatten()
    
    # Plot first 4 channels
    for i in range(min(4, sample.shape[1])):
        axes[i].plot(sample[:, i], linewidth=1)
        axes[i].set_title(f"Channel {i}")
        axes[i].set_xlabel("Time steps")
        axes[i].set_ylabel("Normalized value")
        axes[i].grid(True, alpha=0.3)
    
    plt.suptitle("C-MAPSS FD001 - Sample Time Series (First 4 Channels)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"Sample shape: {sample.shape}")
    print(f"Lookback: {data['lookback']}, Horizon: {data['horizon']}")
else:
    print("C-MAPSS processed data not found!")